# IndianLegal-LLM — QLoRA fine-tune (cloud T4)

Fine-tunes an **Apache-2.0** base into an Indian-law assistant whose outputs pass the
Hour-4 **citation guard** (verbatim quote + `[chunk_id]` + paragraph pinpoint). Only a
small **LoRA adapter** (50–200 MB, MIT) is exported — we never redistribute base weights.

**Base (verified Apache-2.0 on the HF model card):**
- Train: `unsloth/Qwen3-4B-Instruct-2507-bnb-4bit` (bnb-4bit, for Unsloth QLoRA on a free T4)
- Serve: `Qwen/Qwen3-4B-Instruct-2507` + the LoRA adapter
- Phi-4 (MIT) stays the zero-shot serving default; this notebook produces the fine-tuned variant.

**Training data = REAL judgments (M-FT1):** built by `data_pipeline` from the on-disk SC
PDF corpus — extractive, citation-grounded (verbatim quote + chunk_id + ¶) with refusal
hard-negatives, every example guard-checked. Same chunking as retrieval (train == inference).

## CLAUDE.md compliance
- **§2 (licensing):** base is Apache-2.0; the adapter we train + ship is MIT. PDF extraction
  uses permissive pdfminer.six (MIT) / pypdf (BSD) only — no AGPL in the dep tree.
- **§5 (cost/bandwidth):** runs **IN THE CLOUD** on a free **T4**. The corpus is read where it
  lives; base weights download **once** on the GPU host; only the 50–200 MB adapter comes down.
- **§6 (green build):** the offline skeleton + deterministic eval gate are unaffected (stub-pinned).

## Free-tier session limits + the bandwidth rule
- Kaggle: ~30 GPU-hours/week, ~12h sessions, T4 16 GB — enough for a 4B QLoRA run.
- QLoRA 4-bit + gradient checkpointing keeps a 4B model within 16 GB.
- **Corpus must be present in the cloud** at `data/sc/` (the tars + metadata) — do NOT download
  it to a laptop. Extraction (PDF→text) and the embed pass run in-cloud.
- `data/`, `adapters/`, `outputs/`, `*.safetensors` are gitignored — nothing here is committed.

In [ ]:
# Install (cloud GPU). Unsloth pulls transformers/peft/trl/bitsandbytes; the
# permissive PDF extractors (MIT/BSD) are the data-prep deps (no AGPL).
%pip install -q "unsloth[colab-new]" "trl" "datasets" "pdfminer.six" "pypdf"
# Install this repo (the tested chunking, citation guard, builder + serving path).
%pip install -q -e ..

In [ ]:
# --- Run configuration ---
TRAIN_MODEL = "unsloth/Qwen3-4B-Instruct-2507-bnb-4bit"  # Apache-2.0, bnb-4bit (QLoRA)
SERVE_BASE  = "Qwen/Qwen3-4B-Instruct-2507"             # Apache-2.0 serving base
ADAPTER_DIR = "adapters/indianlegal-qwen3-4b-lora"       # gitignored; MIT adapter output
MAX_SEQ_LEN = 2048

# SMOKE=True: tiny toy run (one year, capped) for the acceptance smoke train.
# SMOKE=False: full run over the corpus years.
SMOKE = True
YEARS = "2020" if SMOKE else "2018-2026"

### Step 1 — Build the REAL citation-grounded dataset (M-FT1)
`data_pipeline.corpus` extracts + cleans the judgment PDFs into processed text (permissive
pdfminer.six/pypdf; scanned PDFs are quarantined, coverage reported). `data_pipeline.build_finetune`
then mines extractive, guard-passing examples (holding / issue / statutory / multi-¶) plus refusal
hard-negatives (~30%), split BY judgment (no doc leakage). Output = `data/finetune/{train,dev}.jsonl`
in the QLoRA chat schema.

In [ ]:
from data_pipeline import corpus, build_finetune

# 1a) PDF -> cleaned processed text (build-time; permissive MIT/BSD extractor).
extract_args = ["--years", YEARS, "--out", "data/sc/processed"]
if SMOKE:
    extract_args += ["--limit-per-year", "40"]
corpus.main(extract_args)

# 1b) Mine the citation-grounded instruction dataset (+ refusals), guard-checked.
build_finetune.main(["--processed", "data/sc/processed", "--out", "data/finetune"])

### Step 2 — Load the 4-bit base and attach LoRA (QLoRA)
Unsloth loads the bnb-4bit Apache-2.0 base with gradient checkpointing; PEFT adds the LoRA layers.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TRAIN_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,            # QLoRA 4-bit
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",   # fits a 4B model in 16 GB
    random_state=3407,
)

### Step 3 — Format with the Instruct chat template and train
`max_steps` is tiny for the SMOKE run and larger for a real run. We train on `train.jsonl`.

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

raw = load_dataset("json", data_files="data/finetune/train.jsonl", split="train")

def to_text(example):
    # Apply the model's own Instruct chat template to the system/user/assistant turns.
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

dataset = raw.map(to_text, remove_columns=raw.column_names)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=10 if SMOKE else 400,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

### Step 4 — Export ONLY the LoRA adapter (50–200 MB)
We save the adapter + tokenizer, NOT the merged base weights (which we never redistribute).

In [ ]:
import subprocess

model.save_pretrained(ADAPTER_DIR)       # adapter_model.safetensors + adapter_config.json only
tokenizer.save_pretrained(ADAPTER_DIR)
print(subprocess.run(["du", "-sh", ADAPTER_DIR], capture_output=True, text=True).stdout)
print("Adapter (MIT) ready. This is the ONLY artifact meant to leave the cloud host.")

### Step 5 — Serve the fine-tuned variant (base + adapter)
Retrieval indexes the SAME real PDF text (INGESTOR=local-sc); the Hour-4 guard still applies,
so answers are grounded/cited/pinpointed or refused.

In [ ]:
# Equivalent CLI:
#   export LLM=transformers BASE_MODEL=Qwen/Qwen3-4B-Instruct-2507 \
#          LORA_ADAPTER=adapters/indianlegal-qwen3-4b-lora INGESTOR=local-sc VECTOR_BACKEND=e5
#   python -m indianlegal_llm.app.cli "Is privacy a fundamental right in India?"
from indianlegal_llm.config import Settings
from indianlegal_llm.pipeline import build_pipeline

pipe = build_pipeline(Settings(
    llm="transformers", base_model=SERVE_BASE, adapter=ADAPTER_DIR,
    ingestor="local-sc", vector_backend="e5",
))
print("serving:", pipe.llm_backend, "| source:", pipe.source, "| chunks:", pipe.num_chunks)
answer = pipe.answer("Is privacy a fundamental right in India?")
print("refused:", answer.refused)
for c in answer.citations:
    print("-", c.reference)

---
**Do not commit** `data/`, `adapters/`, `outputs/`, or any `*.safetensors` — all gitignored.
Publish the adapter (MIT) to a registry / HF Hub from the cloud host; the repo only tracks
code, configs, and this notebook (outputs cleared).